In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

# ── READ SILVER TABLES ───────────────────────────────────
df_att = spark.read.table("silver_attendance")
df_enr = spark.read.table("silver_enrolments")

# ── DIM_SCHOOL ───────────────────────────────────────────
# Distinct school records from attendance (most complete source)
window = Window.partitionBy('School_Number').orderBy(F.desc('Census_Year'))

# df_att.select('Type_of_School').distinct().show()


dim_school = df_att.withColumn('rn', F.row_number().over(window)) \
    .filter(F.col('rn') == 1) \
    .drop('rn') \
    .select(
        'School_Number',
        'School_Name',
        'Type_of_School',
        'Suburb',
        'Postcode',
        'Latitude',
        'Longitude'
    )

dim_school.write \
    .format('delta') \
    .mode('overwrite') \
    .option('mergeSchema', 'true') \
    .saveAsTable('dim_school')

# ── DIM_YEAR ─────────────────────────────────────────────
dim_year = df_att.select(
    F.col('Census_Year').alias('Year')
).distinct() \
.withColumn('Decade', (F.col('Year') / 10).cast(IntegerType()) * 10) \
.orderBy('Year')

dim_year.write \
    .format('delta') \
    .mode('overwrite') \
    .option('mergeSchema', 'true') \
    .saveAsTable('dim_year')

# ── FACT_ATTENDANCE ──────────────────────────────────────
# Join attendance and enrolments, keep only measurable facts
fact_attendance = df_att.join(
    df_enr,
    (df_att.School_Number == df_enr.School_Number) &
    (df_att.Census_Year == df_enr.Year),
    how='left'
).select(
    df_att.School_Number,
    df_att.Census_Year.alias('Year'),
    df_att.Avg_Attendance_Rate,
    df_att.Reception,
    df_att.Year_1,
    df_att.Year_2,
    df_att.Year_3,
    df_att.Year_4,
    df_att.Year_5,
    df_att.Year_6,
    df_att.Year_7,
    df_att.Year_8,
    df_att.Year_9,
    df_att.Year_10,
    df_att.Year_11,
    df_att.Year_12,
    df_att.Primary_Other,
    df_att.Secondary_Other,
    df_enr.Total_Enrolments
)

fact_attendance.write \
    .format('delta') \
    .mode('overwrite') \
    .option('mergeSchema', 'true') \
    .saveAsTable('fact_attendance')

print("Star schema saved successfully!")
print(f"dim_school rows: {dim_school.count()}")
print(f"dim_year rows: {dim_year.count()}")
print(f"fact_attendance rows: {fact_attendance.count()}")

# Check how many school numbers match between the two tables
fact = spark.read.table("fact_attendance")
dim = spark.read.table("dim_school")

# Schools in fact but not in dim
missing = fact.select('School_Number').distinct() \
    .join(dim.select('School_Number'), on='School_Number', how='left_anti')

print(f"Schools in fact not in dim: {missing.count()}")
missing.show()

# dim_school.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save("/Volumes/workspace/default/raw/export/dim_school1")
# dim_year.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save("/Volumes/workspace/default/raw/export/dim_year")
# fact_attendance.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save("/Volumes/workspace/default/raw/export/fact_attendance")

print("Exports done!")